<a href="https://colab.research.google.com/github/whitebearhands/lineart-lora/blob/main/lineart_lora_a100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q peft bitsandbytes xformers
!pip install -q Pillow datasets huggingface_hub

print("\n✅ 설치 완료!")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("\n✅ 준비 완료! 다음 단계로 진행하세요.")
else:
    print("\n❌ GPU가 감지되지 않습니다. 런타임 → 런타임 유형 변경 → T4 GPU 설정하세요.")

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

MODEL_NAME = "runwayml/stable-diffusion-v1-5"

# 모델 다운로드 (캐시됨 - 두 번째부터는 빠름)
print("모델 다운로드 중... (첫 실행 시 약 5분 소요)")
pipeline = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    safety_checker=None,
)
print("\n✅ 베이스 모델 다운로드 완료!")

# 메모리 절약을 위해 파이프라인 삭제 (학습 시 다시 로드)
del pipeline
torch.cuda.empty_cache()

In [ ]:
from google.colab import drive
drive.mount('/content/drive' , force_remount=True)

!unzip /content/drive/MyDrive/dataset.zip -d /content/train_data

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
import os
from PIL import Image

# ===== 트리거 워드 설정 =====
# 이 단어를 프롬프트에 넣으면 학습된 스타일이 적용됩니다
TRIGGER_WORD = "lineart_style"  # 원하는 단어로 변경 가능
# ============================

print("BLIP 캡셔닝 모델 로딩 중...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to("cuda")

print("캡션 생성 중...\n")

for fname in sorted(os.listdir(TRAIN_DIR)): # Changed PROCESSED_DIR to TRAIN_DIR
    if not fname.endswith('.png'):
        continue

    img_path = os.path.join(TRAIN_DIR, fname) # Changed PROCESSED_DIR to TRAIN_DIR
    img = Image.open(img_path).convert("RGB")

    # BLIP으로 캡션 생성
    inputs = blip_processor(img, return_tensors="pt").to("cuda", torch.float16)
    output = blip_model.generate(**inputs, max_new_tokens=50)
    caption = blip_processor.decode(output[0], skip_special_tokens=True)

    # 트리거 워드를 앞에 추가
    full_caption = f"{TRIGGER_WORD}, {caption}"

    # 같은 이름의 .txt 파일로 저장
    txt_path = img_path.replace('.png', '.txt')
    with open(txt_path, 'w') as f:
        f.write(full_caption)

    print(f"  {fname} → {full_caption}")

# BLIP 모델 메모리 해제
del blip_model, blip_processor
torch.cuda.empty_cache()

print(f"\n✅ 캡셔닝 완료! 트리거 워드: '{TRIGGER_WORD}'")
print("💡 생성 시 프롬프트에 이 단어를 넣으면 학습된 스타일이 적용됩니다.")

In [ ]:
MAX_TRAIN_STEPS = 5000
TRAIN_BATCH_SIZE = 4        # A100 VRAM 40GB, 넉넉함
GRADIENT_ACCUMULATION = 2
LEARNING_RATE = 1e-4
LORA_RANK = 64
LORA_ALPHA = 32
SAVE_EVERY_N_STEPS = 500
# =====================

OUTPUT_DIR = "/content/lora_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"학습 설정:")
print(f"  LoRA Rank: {LORA_RANK}, Alpha: {LORA_ALPHA}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Steps: {MAX_TRAIN_STEPS}")
print(f"  실질 배치: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  출력: {OUTPUT_DIR}")

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py -O /content/train_lora.py
print("✅ 학습 스크립트 준비 완료")

In [ ]:
import json

DATASET_DIR = "/content/hf_dataset"
os.makedirs(os.path.join(DATASET_DIR, "train"), exist_ok=True)

metadata = []
for fname in sorted(os.listdir(TRAIN_DIR)):
    if not fname.endswith('.png'):
        continue

    txt_file = fname.replace('.png', '.txt')
    txt_path = os.path.join(TRAIN_DIR, txt_file)

    if os.path.exists(txt_path):
        with open(txt_path) as f:
            caption = f.read().strip()
    else:
        caption = TRIGGER_WORD

    # 이미지 복사
    import shutil
    shutil.copy2(
        os.path.join(TRAIN_DIR, fname),
        os.path.join(DATASET_DIR, "train", fname)
    )

    metadata.append({"file_name": fname, "text": caption})

# metadata.jsonl 생성
with open(os.path.join(DATASET_DIR, "train", "metadata.jsonl"), 'w') as f:
    for item in metadata:
        f.write(json.dumps(item) + '\n')

print(f"✅ 데이터셋 준비 완료! ({len(metadata)}장)")

In [ ]:
!accelerate launch /content/train_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/content/hf_dataset/train" \
  --output_dir="{OUTPUT_DIR}" \
  --resolution={RESOLUTION} \
  --train_batch_size={TRAIN_BATCH_SIZE} \
  --gradient_accumulation_steps={GRADIENT_ACCUMULATION} \
  --learning_rate={LEARNING_RATE} \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=100 \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --checkpointing_steps={SAVE_EVERY_N_STEPS} \
  --rank={LORA_RANK} \
  --seed=42 \
  --mixed_precision="fp16" \
  --enable_xformers_memory_efficient_attention \
  --caption_column="text"

print("\n" + "="*50)
print("🎉 LoRA 학습 완료!")
print(f"모델 저장 위치: {OUTPUT_DIR}")
print("="*50)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# 베이스 모델 + LoRA 로드
print("모델 로딩 중...")
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# 학습된 LoRA 적용
pipe.load_lora_weights(OUTPUT_DIR)
print("✅ LoRA 로드 완료!")

In [ ]:
# 이미지 생성 테스트

# ===== 프롬프트 수정 영역 =====
test_prompts = [
    f"{TRIGGER_WORD}, a person sitting at a desk working on a computer, clean lines, white background",
    f"{TRIGGER_WORD}, a doctor talking to a patient in a hospital, minimal illustration",
    f"{TRIGGER_WORD}, a chef cooking in a kitchen, black and white line drawing",
    f"{TRIGGER_WORD}, a teacher standing in front of a classroom, simple outline",
]

NEGATIVE_PROMPT = "blurry, low quality, color, watercolor, photorealistic, 3d render"
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5
# ==============================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(test_prompts), figsize=(6 * len(test_prompts), 6))
if len(test_prompts) == 1:
    axes = [axes]

for i, (ax, prompt) in enumerate(zip(axes, test_prompts)):
    print(f"생성 중 ({i+1}/{len(test_prompts)}): {prompt[:60]}...")

    image = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=torch.Generator("cuda").manual_seed(42 + i),
    ).images[0]

    ax.imshow(image)
    # 프롬프트에서 트리거워드 제외하고 표시
    short_prompt = prompt.replace(f"{TRIGGER_WORD}, ", "")[:40]
    ax.set_title(short_prompt, fontsize=9)
    ax.axis('off')

    # 개별 저장
    image.save(f"/content/result_{i:02d}.png")

plt.suptitle(f"LoRA 생성 결과 (트리거: {TRIGGER_WORD})", fontsize=14)
plt.tight_layout()
plt.show()

print("\n✅ 생성 완료! /content/result_*.png 로 저장됨")

In [ ]:
from huggingface_hub import HfApi, login

# 허깅페이스 토큰 입력 (https://huggingface.co/settings/tokens 에서 발급)
login()

api = HfApi()

# 리포지토리 생성 + 업로드
REPO_ID = "whitebearhands/lineart-lora"  # 본인 아이디/모델명으로 수정

api.create_repo(REPO_ID, exist_ok=True)

# LoRA 파일 업로드
api.upload_folder(
    folder_path="/content/lora_output",
    repo_id=REPO_ID,
    commit_message="Upload lineart LoRA"
)

print(f"✅ 업로드 완료! https://huggingface.co/{REPO_ID}")